# SENTINEL-X — Autonomous Operational Intelligence & Risk Detection Platform

**Complete project notebook | Python · NLP · Machine Learning · Risk Analytics · Automation**

Build an evidence-led prototype that ingests operational records, extracts signals, identifies missing information and deadline risks, detects unusual numeric patterns, scores risk transparently, links related records, and exports a human-review report.

> The built-in dataset is synthetic. Scores are configurable prioritisation heuristics, not legal/financial determinations or calibrated probabilities. Verify every alert against original evidence.

## 1. Architecture

Documents/records → ingestion → cleaning/validation → extraction/classification → deadline and data-quality checks → anomaly detection → explainable risk scoring → related-record matching → human review/report.

Production extensions: FastAPI, PostgreSQL, object storage, OCR, n8n orchestration, React dashboard, Docker, access controls and audit logging.

## 2. Install and import dependencies

In [ ]:
%pip install -q pandas numpy matplotlib scikit-learn python-dateutil

In [ ]:
import re, json, hashlib, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
ANALYSIS_DATE = pd.Timestamp("2026-11-16")
OUT = Path("sentinel_x_output")
OUT.mkdir(exist_ok=True)
print("SENTINEL-X ready | analysis date:", ANALYSIS_DATE.date())

## 3. Create a synthetic operational dataset

The records deliberately include missing fields, overdue items, high-value items and unusual patterns for demonstration. Replace them only with authorised, appropriately protected data.

In [ ]:
records = [
{"record_id":"INV-1001","document_type":"invoice","title":"Office equipment invoice","vendor":"Northstar Supplies","amount":48500,"issue_date":"2026-11-01","due_date":"2026-11-12","status":"pending","owner":"Procurement","description":"Invoice for monitors and keyboards. Payment due within 11 days.","reference_id":"PO-5001"},
{"record_id":"INV-1002","document_type":"invoice","title":"Cloud infrastructure invoice","vendor":"CloudWorks","amount":125000,"issue_date":"2026-11-10","due_date":"2026-11-25","status":"pending","owner":"Engineering","description":"Monthly cloud hosting and infrastructure services.","reference_id":"PO-5002"},
{"record_id":"INV-1003","document_type":"invoice","title":"Consulting services invoice","vendor":"","amount":78000,"issue_date":"2026-10-20","due_date":"2026-11-05","status":"pending","owner":"Operations","description":"Consulting services delivered for process improvement.","reference_id":""},
{"record_id":"CTR-2001","document_type":"contract","title":"Software subscription agreement","vendor":"DataSphere","amount":240000,"issue_date":"2026-08-01","due_date":"2026-11-20","status":"active","owner":"Legal","description":"Subscription agreement. Renewal and termination clauses require review.","reference_id":"AGR-2201"},
{"record_id":"CTR-2002","document_type":"contract","title":"Analytics platform agreement","vendor":"InsightGrid","amount":98000,"issue_date":"2026-10-15","due_date":"","status":"under_review","owner":"Legal","description":"Agreement for analytics platform implementation. Signature pending.","reference_id":"AGR-2202"},
{"record_id":"PO-5001","document_type":"purchase_order","title":"Office equipment purchase order","vendor":"Northstar Supplies","amount":47000,"issue_date":"2026-10-28","due_date":"2026-11-30","status":"approved","owner":"Procurement","description":"Purchase order for monitors and keyboards.","reference_id":"REQ-7001"},
{"record_id":"PO-5002","document_type":"purchase_order","title":"Cloud services purchase order","vendor":"CloudWorks","amount":120000,"issue_date":"2026-11-08","due_date":"2026-12-08","status":"approved","owner":"Engineering","description":"Purchase order for cloud hosting services.","reference_id":"REQ-7002"},
{"record_id":"RPT-3001","document_type":"project_report","title":"Migration project status","vendor":"","amount":0,"issue_date":"2026-11-14","due_date":"2026-11-18","status":"in_progress","owner":"Engineering","description":"Migration milestone delayed. External data transfer dependency unresolved.","reference_id":"PRJ-9001"},
{"record_id":"INV-1004","document_type":"invoice","title":"Specialist equipment invoice","vendor":"Apex Systems","amount":980000,"issue_date":"2026-11-12","due_date":"2026-12-12","status":"pending","owner":"Procurement","description":"High-value specialist equipment invoice for laboratory infrastructure.","reference_id":"PO-5999"},
{"record_id":"INV-1005","document_type":"invoice","title":"Facilities maintenance invoice","vendor":"Metro Facilities","amount":32000,"issue_date":"2026-11-13","due_date":"2026-11-28","status":"paid","owner":"Administration","description":"Monthly facilities maintenance and repair services.","reference_id":"PO-5010"}
]
df = pd.DataFrame(records)
df["synthetic_data"] = True
df["source_hash"] = df.apply(lambda r: hashlib.sha256((r["title"]+r["description"]).encode()).hexdigest(), axis=1)
display(df)

## 4. Clean and validate records

In [ ]:
def clean_text(x):
    if pd.isna(x): return ""
    return re.sub(r"\\s+", " ", str(x)).strip()

for c in ["record_id","document_type","title","vendor","status","owner","description","reference_id"]:
    df[c] = df[c].apply(clean_text)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["issue_date"] = pd.to_datetime(df["issue_date"], errors="coerce")
df["due_date"] = pd.to_datetime(df["due_date"], errors="coerce")
quality = pd.DataFrame({"missing":df.isna().sum(),"missing_pct":(df.isna().mean()*100).round(1)})
display(quality)
print("Duplicate record IDs:", df["record_id"].duplicated().sum())

## 5. Rule-based document classification baseline

This is a transparent baseline. A production classifier needs labelled examples and evaluation.

In [ ]:
KEYWORDS = {
"invoice":["invoice","payment","bill","amount due"],
"contract":["agreement","contract","termination","renewal","signature"],
"purchase_order":["purchase order","procurement"],
"project_report":["project","milestone","migration","progress","status"]
}
def classify(text):
    t = str(text).lower()
    scores = {k:sum(term in t for term in v) for k,v in KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] else "unclassified"
df["predicted_type"] = (df["title"]+" "+df["description"]).apply(classify)
display(df[["record_id","document_type","predicted_type"]])

## 6. Extract operational signals

In [ ]:
TERMS = {
"delay":["delay","delayed","overdue","pending"],
"dependency":["dependency","dependent","external"],
"legal_review":["termination","renewal","legal","agreement"],
"payment":["payment","invoice","amount due"],
"signature":["signature","unsigned","signing"]
}
def extract_signals(text):
    t = str(text).lower()
    return {k:any(term in t for term in vals) for k,vals in TERMS.items()}
signals = pd.DataFrame([(extract_signals(a+" "+b)) for a,b in zip(df.title,df.description)])
df = pd.concat([df, signals.add_prefix("signal_")], axis=1)
display(df[["record_id"]+[f"signal_{k}" for k in TERMS]])

## 7. Missing-field and deadline checks

In [ ]:
REQUIRED = {
"invoice":["vendor","amount","issue_date","due_date","reference_id"],
"contract":["vendor","issue_date","due_date","reference_id"],
"purchase_order":["vendor","amount","issue_date","reference_id"],
"project_report":["issue_date","due_date","owner","reference_id"]
}
def field_issues(row):
    issues=[]
    for field in REQUIRED.get(row.document_type, []):
        val=row[field]
        if pd.isna(val) or str(val).strip()=="":
            issues.append("Missing required field: "+field)
    return issues
df["field_issues"] = df.apply(field_issues,axis=1)
df["missing_field_count"] = df.field_issues.str.len()
df["days_to_due"] = (df.due_date-ANALYSIS_DATE).dt.days
closed = df.status.str.lower().isin(["paid","closed","completed"])
df["is_overdue"] = df.due_date.notna() & (df.days_to_due<0) & ~closed
df["deadline_soon"] = df.due_date.notna() & df.days_to_due.between(0,7) & ~closed
display(df[["record_id","field_issues","days_to_due","is_overdue","deadline_soon"]])

## 8. Numerical anomaly detection

Isolation Forest is a screening method, not proof of fraud or error. In real operations, compare records within appropriate document categories and validate alerts against historical outcomes.

In [ ]:
features = pd.DataFrame({
"amount":df.amount.fillna(0),
"days_to_due":df.days_to_due.fillna(0),
"missing_field_count":df.missing_field_count,
"description_length":df.description.str.len()
})
if len(df)>=10 and features.nunique().max()>1:
    X=StandardScaler().fit_transform(features)
    model=IsolationForest(n_estimators=200,contamination=.15,random_state=RANDOM_STATE)
    df["is_anomaly"]=model.fit_predict(X)==-1
else:
    df["is_anomaly"]=False
display(df[["record_id","amount","is_anomaly"]])

## 9. Explainable risk scoring

The score is a configurable prioritisation heuristic. It is not a probability. Each contribution is retained as an explanation.

In [ ]:
def risk_assessment(row):
    score, reasons = 0, []
    if row.is_overdue: score+=35; reasons.append("Overdue deadline")
    elif row.deadline_soon: score+=15; reasons.append("Deadline within 7 days")
    if row.missing_field_count:
        pts=min(10*int(row.missing_field_count),30)
        score+=pts; reasons.append(f"{int(row.missing_field_count)} required field(s) missing")
    if row.is_anomaly: score+=20; reasons.append("Unusual numeric/operational pattern")
    if row.signal_delay: score+=10; reasons.append("Delay-related language")
    if row.signal_dependency: score+=5; reasons.append("Dependency-related language")
    if row.signal_legal_review: score+=10; reasons.append("Contract/legal-review language")
    if row.amount>=500000: score+=15; reasons.append("High-value threshold exceeded")
    score=min(score,100)
    level="CRITICAL" if score>=80 else "HIGH" if score>=60 else "MODERATE" if score>=35 else "LOW"
    return pd.Series({"risk_score":score,"risk_level":level,"risk_reasons":reasons})
df=pd.concat([df,df.apply(risk_assessment,axis=1)],axis=1)
display(df[["record_id","risk_score","risk_level","risk_reasons"]].sort_values("risk_score",ascending=False))

## 10. Related-record detection with TF-IDF similarity

Similarity suggests candidate links only. It does not prove that records refer to the same transaction.

In [ ]:
corpus=(df.title.fillna("")+" "+df.description.fillna("")).tolist()
vec=TfidfVectorizer(stop_words="english",ngram_range=(1,2))
mat=vec.fit_transform(corpus)
sim=cosine_similarity(mat)
pairs=[]
for i in range(len(df)):
    for j in range(i+1,len(df)):
        same_vendor=bool(df.iloc[i].vendor and df.iloc[i].vendor.lower()==df.iloc[j].vendor.lower())
        linked=(df.iloc[i].reference_id==df.iloc[j].record_id or df.iloc[j].reference_id==df.iloc[i].record_id)
        if sim[i,j]>=.12 or same_vendor or linked:
            pairs.append({"record_a":df.iloc[i].record_id,"record_b":df.iloc[j].record_id,
                          "similarity":round(float(sim[i,j]),3),"same_vendor":same_vendor,"linked_reference":linked})
related=pd.DataFrame(pairs)
display(related.sort_values("similarity",ascending=False) if len(related) else related)

## 11. Visualise risk distribution

In [ ]:
order=["LOW","MODERATE","HIGH","CRITICAL"]
df.risk_level.value_counts().reindex(order,fill_value=0).plot(kind="bar",figsize=(8,4),title="Records by risk level")
plt.xlabel("Risk level"); plt.ylabel("Record count"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9,4))
plt.bar(df.record_id,df.risk_score)
plt.title("Explainable risk score by record"); plt.xlabel("Record ID"); plt.ylabel("Score (0–100)")
plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()

## 12. Human review queue

The platform prioritises review; it must not automatically reject invoices, terminate contracts or release payments.

In [ ]:
review_cols=["record_id","document_type","title","owner","amount","due_date","status","risk_score","risk_level","risk_reasons"]
review=df[df.risk_level.isin(["CRITICAL","HIGH","MODERATE"])].sort_values("risk_score",ascending=False)
display(review[review_cols])

## 13. Generate an evidence-linked report

In [ ]:
def generate_report(data):
    items=[]
    for _,r in data.sort_values("risk_score",ascending=False).iterrows():
        items.append({
            "record_id":r.record_id,"title":r.title,"risk_score":int(r.risk_score),
            "risk_level":r.risk_level,"reasons":r.risk_reasons,"owner":r.owner,
            "due_date":r.due_date.strftime("%Y-%m-%d") if pd.notna(r.due_date) else None,
            "recommended_action":"Human review: verify the original document and linked records."
        })
    return {"platform":"SENTINEL-X","analysis_date":str(ANALYSIS_DATE.date()),
            "synthetic_demo_data":True,"record_count":len(data),
            "risk_distribution":{str(k):int(v) for k,v in data.risk_level.value_counts().items()},
            "review_items":items,
            "disclaimer":"Synthetic demonstration. Risk scores are heuristic review priorities, not confirmed findings."}
report=generate_report(df)
report_path=OUT/"sentinel_x_risk_report.json"
report_path.write_text(json.dumps(report,indent=2,default=str),encoding="utf-8")
print(report_path.resolve())
print(json.dumps(report,indent=2)[:4000])

## 14. Export processed records and review queue

In [ ]:
export=df.copy()
export["issue_date"]=export.issue_date.astype("string")
export["due_date"]=export.due_date.astype("string")
export["field_issues"]=export.field_issues.apply(json.dumps)
export["risk_reasons"]=export.risk_reasons.apply(json.dumps)
records_path=OUT/"sentinel_x_processed_records.csv"
queue_path=OUT/"sentinel_x_review_queue.csv"
export.to_csv(records_path,index=False)
review[review_cols].to_csv(queue_path,index=False)
print("Created:")
print(records_path.resolve())
print(queue_path.resolve())
print(report_path.resolve())

## 15. Optional: load an authorised CSV

Expected fields: `record_id`, `document_type`, `title`, `vendor`, `amount`, `issue_date`, `due_date`, `status`, `owner`, `description`, `reference_id`.

Only use data you are authorised to process. Remove personal information and secrets, and follow organisational retention and access rules.

In [ ]:
# Uncomment and update the path:
# custom_df = pd.read_csv("your_authorised_operational_records.csv")
# display(custom_df.head())

## 16. Production implementation blueprint

### Document ingestion
- PDF: PyMuPDF or pdfplumber.
- DOCX: python-docx.
- Scanned files: OCR (e.g. Tesseract or managed OCR).
- Preserve page numbers and source locations for every extracted field.

### Backend and storage
- FastAPI endpoints: `/documents`, `/risk/queue`, `/risk/{id}`, `/analytics/summary`, `/relationships`, `/reports`.
- PostgreSQL for metadata, extracted fields, risk events and audit history.
- Object storage for originals.
- pgvector for semantic retrieval; Neo4j for rich relationship exploration if justified.

### Automation
- n8n detects new/modified Drive files.
- A secure processing service receives file references.
- Workers extract and validate fields.
- Database stores processing state and risk events.
- Approved alerts enter a human review queue.
- Idempotency keys prevent duplicate processing.

### Machine learning and evaluation
- Label a representative dataset before training a document classifier.
- Measure extraction precision, recall and field-level exact match.
- Validate anomaly alerts against reviewed historical outcomes.
- Calibrate risk scores before calling them predictive.
- Track false positives, missed risks, latency and data quality.

## 17. Suggested API design

| Method | Endpoint | Purpose |
|---|---|---|
| POST | `/api/v1/documents` | Register document |
| GET | `/api/v1/documents` | Search documents |
| GET | `/api/v1/documents/{id}` | Document details |
| POST | `/api/v1/documents/{id}/process` | Start processing |
| GET | `/api/v1/risk/queue` | Human review queue |
| GET | `/api/v1/risk/{id}` | Risk explanation |
| GET | `/api/v1/analytics/summary` | Dashboard metrics |
| GET | `/api/v1/relationships/{id}` | Related records |
| POST | `/api/v1/reports` | Generate report |
| POST | `/api/v1/reviews/{id}` | Save reviewer decision |

### Suggested frontend
Overview · Document Explorer · Risk Radar · Deadline Monitor · Related Records · Reports · Pipeline Monitor · Review Queue.

## 18. Testing checklist

- [ ] Duplicate uploads do not create duplicate records.
- [ ] Missing/corrupt files fail safely.
- [ ] OCR retains page references.
- [ ] Dates and currencies are parsed correctly.
- [ ] Required fields vary by document type.
- [ ] Paid/closed records are not flagged overdue.
- [ ] Risk explanations show every contributing rule.
- [ ] Anomaly alerts are validated on adequate historical data.
- [ ] Reviewers can accept, dismiss or escalate alerts.
- [ ] Human decisions are audit logged.
- [ ] Secrets and documents are stored securely.

## 19. Eight-week development roadmap

| Week | Work | Deliverable |
|---|---|---|
| 1 | Python, schema, synthetic dataset | Working notebook |
| 2 | PDF/DOCX/CSV ingestion and OCR | Ingestion service |
| 3 | NLP extraction and classification | Structured records |
| 4 | Risk rules and deadline intelligence | Review queue |
| 5 | Anomaly detection and evaluation | Risk analytics |
| 6 | PostgreSQL, FastAPI and n8n | Automated backend |
| 7 | React dashboard and reporting | Web application |
| 8 | Testing, Docker and deployment | Portfolio MVP |

## 20. Suggested repository structure

```text
SENTINEL-X/
├── backend/app/{api,core,models,schemas,repositories,services}/
├── ingestion/{connectors,parsers,ocr,validators}/
├── intelligence/{classification,extraction,anomaly_detection,risk_engine,relationship_engine}/
├── frontend/
├── database/
├── automation/
├── notebooks/
├── tests/
├── docs/
├── docker-compose.yml
├── .env.example
├── .gitignore
└── README.md
```

## 21. Portfolio description

**SENTINEL-X — Autonomous Operational Intelligence & Risk Detection Platform**

An AI-assisted operational intelligence platform designed to transform business documents and records into explainable risk signals. Combines document processing, NLP-based information extraction, rule-driven risk assessment, anomaly detection, relationship analysis and workflow automation to prioritise records for human review.

**Engineering principle:** Every alert must be explainable, traceable to source evidence and reviewable by a human. A heuristic score must never be presented as a confirmed finding or guaranteed prediction.